# Evaluation on Synthbuster + RAISE-1k (Section III-D), on Kaggle — CLIP replication

Set `BACKBONE` in the first code cell (`"clip"` here). Inputs: the **CLIP training output** and the **prep_synthbuster output**. Recompression is off for CLIP (H3 is tested on DINOv2). Also evaluates UnivFD's released detector unchanged (reproduction check of Table I). About 1 h on T4 ×2.

Runs `eval_local.py` on both T4 GPUs in parallel (the LoRA models are split between them), then computes the full analysis:
per-run metrics, Δ(LP), Δ(LoRA), I, R with seed-level t-intervals and decision categories, the paired bootstrap, the
learning-rate sensitivity of I, and the recompression shifts.

**Inputs (Add Input):**
1. the output of the **train-v7** notebook (contains `thesis_runs/` with `protocol_frozen.json`),
2. the output of the **prep_synthbuster** notebook (contains `synthbuster_336/manifest.csv`).

**Session:** GPU T4 ×2, Internet On. About 3–4 h. Results in `/kaggle/working/results/` (`summary.txt` first).
If a session is cut short, commit again with this notebook's previous output added as input: finished models are reused.

In [ ]:
import os, sys, json, time, shutil, subprocess, importlib, urllib.request
from pathlib import Path

NTFY_TOPIC = ""   # live progress at https://ntfy.sh/<topic>; "" turns it off
INPUT_ROOT, WORK = Path("/kaggle/input"), Path("/kaggle/working")
OUT = WORK / "results"
BATCH, WORKERS = 32, 2
BACKBONE = "clip"            # which training run to evaluate ("dinov2" or "clip")
RECOMPRESSION = False        # CLIP replication: original images only


def notify(msg):
    if NTFY_TOPIC:
        try:
            urllib.request.urlopen(urllib.request.Request(f"https://ntfy.sh/{NTFY_TOPIC}", data=str(msg)[:3500].encode(),
                                                          method="POST"), timeout=10).read()
        except Exception as e:
            print("notify failed:", type(e).__name__)


def find(root, name, max_depth=8):
    """Folders containing a file called `name` (symlink-loop safe, skips image folders)."""
    out, seen = [], set()
    for dp, dns, fns in os.walk(root, followlinks=True):
        real = os.path.realpath(dp)
        if real in seen or dp[len(str(root)):].count(os.sep) > max_depth:
            dns[:] = []; continue
        seen.add(real)
        if name in fns:
            out.append(Path(dp))
        dns[:] = [d for d in dns if not d.startswith(("orig", "jpeg_q", "lora", "lp", "real", "fake", "_tmp"))]
    return out


notify("eval session started")
# PEFT refuses to load LoRA when torchao < 0.16 is installed (Kaggle ships 0.10); nothing here uses torchao
import importlib.metadata as md
from packaging.version import Version
try:
    if Version(md.version("torchao")) < Version("0.16.0"):
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=True)
        importlib.invalidate_caches(); print("removed incompatible torchao")
except md.PackageNotFoundError:
    pass

runs = [p for p in find(INPUT_ROOT, f"protocol_frozen_{BACKBONE}.json")] or [p for p in find(INPUT_ROOT, "protocol_frozen.json")]
data = [p for p in find(INPUT_ROOT, "manifest.csv") if (p / "orig").is_dir()]
assert len(runs) >= 1, f"add the training output for {BACKBONE} as input (thesis_runs with its protocol_frozen file)"
assert len(data) == 1, f"add the prep_synthbuster output as input; found {data}"
RUNS, DATA = runs[0], data[0]
print("runs:", RUNS, "\ndata:", DATA)
fz_path = RUNS / f"protocol_frozen_{BACKBONE}.json"
fz = json.loads((fz_path if fz_path.exists() else RUNS / "protocol_frozen.json").read_text())
print("protocol", fz["fingerprint"], "frozen", fz["frozen_utc"])

# predictions from an earlier session of this notebook (added as input) are reused
for prev in find(INPUT_ROOT, "_index.npz"):
    (OUT / "preds").mkdir(parents=True, exist_ok=True)
    n = 0
    for f in prev.iterdir():
        if f.is_file() and not (OUT / "preds" / f.name).exists():
            shutil.copy2(f, OUT / "preds" / f.name); n += 1
    print(f"reused {n} prediction files from {prev}")

from huggingface_hub import snapshot_download
snapshot_download(fz["protocol"]["weights"][BACKBONE], allow_patterns=["*.json", "*.safetensors"])

# UnivFD's released detector head (a single linear layer) for the reproduction check of Table I
UNIVFD_URL = "https://github.com/WisconsinAIVision/UniversalFakeDetect/raw/main/pretrained_weights/fc_weights.pth"
UNIVFD_HEAD = WORK / "univfd_fc_weights.pth"
try:
    if BACKBONE == "clip" and not UNIVFD_HEAD.exists():
        urllib.request.urlretrieve(UNIVFD_URL, UNIVFD_HEAD)
    print("UnivFD head:", UNIVFD_HEAD.stat().st_size, "bytes")
except Exception as e:
    UNIVFD_HEAD = None
    print("UnivFD head not available, reproduction skipped:", type(e).__name__, e)
import torch
N_GPUS = torch.cuda.device_count()
print("GPUs:", N_GPUS)
notify(f"eval setup OK: {N_GPUS} GPUs")

In [ ]:
%%writefile /kaggle/working/eval_local.py
"""Evaluate all trained runs on Synthbuster (+ RAISE-1k) and compute the paper's analysis (Sections III-A, III-D).

Inputs
  --runs  the thesis_runs folder from the Kaggle training output (download the output, unzip)
  --data  the folder made by prep_synthbuster_local.py (has manifest.csv, orig/, jpeg_q*/)
  --out   a results folder (created)

Setup (once), same Python environment as your PyTorch with CUDA:
    pip install "transformers>=5" peft numpy pandas pillow scipy scikit-learn
  transformers 5.x is required: the adapters were trained with it (layer names q_proj/k_proj/...).

Run (Windows PowerShell example, one line):
    python eval_local.py --runs "D:\\thesis\\thesis_runs" --data "D:\\thesis\\synthbuster_336" --out "D:\\thesis\\results"

It can be stopped and restarted: finished models are skipped (predictions are cached in --out/preds).
Quick test first: add  --limit 50  (50 images per generator; results go to the same --out, so use a test folder).

What is evaluated (paper III-D)
  * LP: all 30 heads (3 LRs x 5 seeds x 2 versions) on the original images and on the six recompressed copies.
    One backbone pass serves every head.
  * LoRA at the selected LR: 5 seeds x 2 versions, original + six recompressed copies.
  * LoRA grid runs at the other LRs (first seed): original images only (sensitivity analysis).
Outputs in --out: metrics_runs.csv, metrics_per_generator.csv, recompression.csv, contrasts.csv,
  bootstrap.csv, sensitivity_I.csv, summary.txt, and preds/*.npz (per-image posteriors).
"""
import argparse, json, math, sys, time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from scipy import stats
from sklearn.metrics import roc_auc_score

QUALITIES = [50, 60, 70, 80, 90, 95]
CROP = 336
NORMS = {"dinov2": ((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
         "clip": ((0.48145466, 0.4578275, 0.40821073), (0.26862954, 0.26130258, 0.27577711))}
MEAN = torch.tensor(NORMS["dinov2"][0]).view(1, 3, 1, 1)
STD = torch.tensor(NORMS["dinov2"][1]).view(1, 3, 1, 1)


FWD_KW = {}


def set_norm(backbone):
    """Normalization and forward arguments per backbone (224-px CLIP: position embeddings interpolated to 336)."""
    global MEAN, STD, FWD_KW
    m, s = NORMS["clip"] if "clip" in backbone else NORMS["dinov2"]
    MEAN, STD = torch.tensor(m).view(1, 3, 1, 1), torch.tensor(s).view(1, 3, 1, 1)
    FWD_KW = {"interpolate_pos_encoding": True} if "clip" in backbone else {}
THRESHOLD = 0.03          # practical threshold of the decision rule (III-A)
N_BOOT = 10_000
DEV = "cuda" if torch.cuda.is_available() else "cpu"


def log(msg):
    print(time.strftime("%H:%M:%S"), msg, flush=True)


# ----------------------------------------------------------------------------- data
class Images(torch.utils.data.Dataset):
    def __init__(self, paths):
        self.paths = paths

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        with Image.open(self.paths[i]) as im:
            a = np.array(im.convert("RGB"))
        if a.shape != (CROP, CROP, 3):
            raise ValueError(f"{self.paths[i]}: {a.shape}")
        return torch.from_numpy(a).permute(2, 0, 1)


def load_eval_set(data, limit):
    man = pd.read_csv(data / "manifest.csv")
    man = man[~man["excluded"].astype(bool)].copy()
    if limit:
        man = man.groupby("generator", group_keys=False).head(limit)
    man = man.sort_values(["label", "generator", "id"]).reset_index(drop=True)
    paths = {"orig": [str(data / p) for p in man["path"]]}
    for q in QUALITIES:
        paths[f"q{q}"] = [str(data / f"jpeg_q{q}" / (p.replace("orig/", "", 1)[:-4] + ".jpg")) for p in man["path"]]
    missing = [p for v in paths.values() for p in v[:3] if not Path(p).is_file()]
    if missing:
        sys.exit(f"missing evaluation files, e.g. {missing[:3]}")
    return man, paths


# ----------------------------------------------------------------------------- models
def load_backbone(name, weights):
    from transformers import Dinov2Config, Dinov2Model
    if name == "clip":
        from transformers import CLIPVisionModel
        return CLIPVisionModel.from_pretrained(weights, attn_implementation="sdpa")
    if name == "tiny-clip-test":
        from transformers import CLIPVisionConfig, CLIPVisionModel
        torch.manual_seed(12345)
        return CLIPVisionModel(CLIPVisionConfig(hidden_size=64, intermediate_size=128, num_hidden_layers=2,
                                                num_attention_heads=4, image_size=224, patch_size=14))
    if name == "tiny-test":          # offline test only (same fixed random weights as thesis_lib)
        torch.manual_seed(12345)
        return Dinov2Model(Dinov2Config(hidden_size=64, num_hidden_layers=2, num_attention_heads=4,
                                        intermediate_size=128, patch_size=14, image_size=518))
    return Dinov2Model.from_pretrained(weights, attn_implementation="sdpa")


@torch.no_grad()
def cls_features(backbone, paths, batch, workers):
    backbone.eval().to(DEV)
    dl = torch.utils.data.DataLoader(Images(paths), batch_size=batch, num_workers=workers, pin_memory=DEV == "cuda")
    out, t0 = [], time.time()
    for i, x in enumerate(dl):
        x = (x.to(DEV).float() / 255 - MEAN.to(DEV)) / STD.to(DEV)
        with torch.autocast("cuda", dtype=torch.float16, enabled=DEV == "cuda"):
            out.append(backbone(pixel_values=x, **FWD_KW).pooler_output.float().cpu())
        if i % 100 == 0 and i:
            done = (i + 1) * batch
            log(f"    {done}/{len(paths)} images ({done / (time.time() - t0):.1f} img/s)")
    return torch.cat(out)


def head_from(path, dim):
    h = nn.Linear(dim, 1)
    h.load_state_dict(torch.load(path, map_location="cpu"))
    return h.eval()


def posteriors(head, feats):
    with torch.no_grad():
        return torch.sigmoid(head(feats).squeeze(1)).numpy()


@torch.no_grad()
def univfd_posteriors(weights, head_path, paths, batch, workers, tiny=False):
    """UnivFD [9] exactly as released: frozen CLIP ViT-L/14, its ProGAN-trained linear layer on the 768-d image
    embedding, 224 center crop without resizing (the centre of our 336 crop), CLIP normalization."""
    from transformers import CLIPVisionConfig, CLIPVisionModelWithProjection
    if tiny:
        torch.manual_seed(12345)
        model = CLIPVisionModelWithProjection(CLIPVisionConfig(hidden_size=64, intermediate_size=128, num_hidden_layers=2,
                                                               num_attention_heads=4, image_size=224, patch_size=14,
                                                               projection_dim=32))
    else:
        model = CLIPVisionModelWithProjection.from_pretrained(weights, attn_implementation="sdpa")
    model.eval().to(DEV)
    sd = torch.load(head_path, map_location="cpu")
    sd = {k.split("fc.")[-1]: v for k, v in sd.items()}           # accept "fc.weight" or "weight"
    fc = nn.Linear(sd["weight"].shape[1], 1)
    fc.load_state_dict(sd)
    mean = torch.tensor(NORMS["clip"][0], device=DEV).view(1, 3, 1, 1)
    std = torch.tensor(NORMS["clip"][1], device=DEV).view(1, 3, 1, 1)
    dl = torch.utils.data.DataLoader(Images(paths), batch_size=batch, num_workers=workers, pin_memory=DEV == "cuda")
    out = []
    for x in dl:
        o = (CROP - 224) // 2
        x = (x[:, :, o:o + 224, o:o + 224].to(DEV).float() / 255 - mean) / std
        with torch.autocast("cuda", dtype=torch.float16, enabled=DEV == "cuda"):
            e = model(pixel_values=x).image_embeds.float().cpu()
        out.append(torch.sigmoid(fc(e).squeeze(1)))
    return torch.cat(out).numpy()


# ----------------------------------------------------------------------------- metrics
def bacc(y, p, t=0.5):
    return 0.5 * ((p[y == 1] >= t).mean() + (p[y == 0] < t).mean())


def run_metrics(y, p, gen, thr):
    m = {"bacc": bacc(y, p), "tpr": float((p[y == 1] >= 0.5).mean()), "tnr": float((p[y == 0] < 0.5).mean()),
         "auroc": float(roc_auc_score(y, p)), "bacc_valthr": bacc(y, p, thr), "val_threshold": thr}
    per_gen = {g: float((p[gen == g] >= 0.5).mean()) for g in np.unique(gen[y == 1])}   # TPR per generator
    return m, per_gen


def t_interval(x):
    x = np.asarray(x, float)
    n, mean, sd = len(x), x.mean(), x.std(ddof=1) if len(x) > 1 else float("nan")
    half = stats.t.ppf(0.975, n - 1) * sd / math.sqrt(n) if n > 1 else float("nan")
    mde = (stats.t.ppf(0.975, n - 1) + stats.t.ppf(0.80, n - 1)) * sd / math.sqrt(n) if n > 1 else float("nan")
    return mean, sd, mean - half, mean + half, mde


def category(est, lo, hi, thr=THRESHOLD):
    """Five-way decision rule of Section III-A (0.03 is a practical threshold, not a significance test)."""
    if lo > 0:
        return "supported" if est >= thr else "detectable, below threshold"
    if hi < 0:
        return "reversed"
    return "near zero" if (lo >= -thr and hi <= thr) else "inconclusive"


# ----------------------------------------------------------------------------- main
def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--runs", required=True)
    ap.add_argument("--data", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--batch", type=int, default=32)
    ap.add_argument("--workers", type=int, default=0, help="DataLoader workers (0 is safest on Windows)")
    ap.add_argument("--limit", type=int, default=None)
    ap.add_argument("--backbone", default="dinov2")
    ap.add_argument("--shard", type=int, default=0, help="this process's share of the LoRA models (0-based)")
    ap.add_argument("--nshards", type=int, default=1, help="number of processes sharing the work (e.g. one per GPU)")
    ap.add_argument("--skip-analysis", action="store_true", help="only compute predictions (used by sharded runs)")
    ap.add_argument("--no-recompression", action="store_true", help="original images only (skips the s(c,q) diagnostic)")
    ap.add_argument("--univfd-head", default=None, help="UnivFD fc_weights.pth: also evaluate the released UnivFD detector")
    args = ap.parse_args()
    runs, data, out = Path(args.runs), Path(args.data), Path(args.out)
    (out / "preds").mkdir(parents=True, exist_ok=True)

    set_norm(args.backbone)
    fz_path = runs / f"protocol_frozen_{args.backbone}.json"
    frozen = json.loads((fz_path if fz_path.exists() else runs / "protocol_frozen.json").read_text())
    P = frozen["protocol"]
    weights = P["weights"].get(args.backbone, P["weights"]["dinov2"])
    seeds, s0, versions = P["seeds"], P["first_seed"], P["versions"]
    sel = {s: json.loads((runs / f"selection_{args.backbone}_{s}.json").read_text())["selected_lr"] for s in ("lp", "lora")}
    log(f"protocol frozen {frozen['frozen_utc']} ({frozen['fingerprint']}); selected LP lr {sel['lp']:g}, LoRA lr {sel['lora']:g}")

    man, paths = load_eval_set(data, args.limit)
    if args.no_recompression:
        paths = {"orig": paths["orig"]}
    y, gen = man["label"].to_numpy(), man["generator"].to_numpy()
    np.savez(out / "preds" / "_index.npz", id=man["id"].to_numpy(), label=y, generator=gen)
    (out / "preds").joinpath("_backbone.txt").write_text(args.backbone)
    log(f"evaluation set: {int((y == 1).sum())} synthetic in {len(np.unique(gen[y == 1]))} generators, "
        f"{int((y == 0).sum())} authentic; versions: {', '.join(paths)}")

    def metrics_of(name):
        return json.loads((runs / ("lp" if "_lp_" in name else "lora") / name / "metrics.json").read_text())

    # ---------- LP: one backbone pass, then every head
    fpath = out / "preds" / f"_backbone_features_{args.backbone}.pt"
    if args.shard != 0:
        feats = None          # the LP part is done by shard 0
    elif fpath.exists():
        feats = torch.load(fpath)
    else:
        bb = load_backbone(args.backbone, weights)
        feats = {}
        for v, ps in paths.items():
            log(f"backbone features ({v})")
            feats[v] = cls_features(bb, ps, args.batch, args.workers)
        torch.save(feats, fpath)
        del bb
    for d in sorted((runs / "lp").glob(f"{args.backbone}_lp_*")) if feats is not None else []:
        dim = feats["orig"].shape[1]
        f = out / "preds" / f"{d.name}.npz"
        if not f.exists() and (d / "head.pt").exists():
            h = head_from(d / "head.pt", dim)
            np.savez(f, **{v: posteriors(h, feats[v]) for v in paths})
    log("LP heads evaluated")

    # ---------- LoRA
    from peft import PeftModel
    todo = []
    for d in sorted((runs / "lora").glob(f"{args.backbone}_lora_*")):
        m = metrics_of(d.name)
        full = m["lr"] == sel["lora"]
        todo.append((d, list(paths) if full else ["orig"]))
    for i, (d, vs) in enumerate(todo):
        f = out / "preds" / f"{d.name}.npz"
        if f.exists() or i % args.nshards != args.shard:
            continue
        log(f"LoRA {i + 1}/{len(todo)}: {d.name} ({'full' if len(vs) > 1 else 'original only'})")
        model = PeftModel.from_pretrained(load_backbone(args.backbone, weights), str(d / "adapter")).merge_and_unload()
        h = head_from(d / "head.pt", model.config.hidden_size)
        res = {}
        for v in vs:
            res[v] = posteriors(h, cls_features(model, paths[v], args.batch, args.workers))
        np.savez(f, **res)
        del model
        if DEV == "cuda":
            torch.cuda.empty_cache()

    univfd_line = []
    if args.univfd_head and args.shard == 0:
        f = out / "preds" / "_univfd.npz"
        if not f.exists():
            log("UnivFD (released head, 224 center crop)")
            np.savez(f, orig=univfd_posteriors(weights, args.univfd_head, paths["orig"], args.batch, args.workers,
                                               tiny=args.backbone.startswith("tiny")))
    if args.skip_analysis:
        log("predictions done (analysis skipped)")
        return
    if (out / "preds" / "_univfd.npz").exists():
        pu = np.load(out / "preds" / "_univfd.npz")["orig"]
        pu_pred = pu > 0.5                                  # UnivFD's rule: fake if p > 0.5
        tpr_u, tnr_u = float(pu_pred[y == 1].mean()), float((~pu_pred[y == 0]).mean())
        gen_u = {g: float(pu_pred[(gen == g)].mean()) for g in np.unique(gen[y == 1])}
        pd.DataFrame([{"detector": "UnivFD (released)", "bacc": (tpr_u + tnr_u) / 2, "tpr": tpr_u, "tnr": tnr_u,
                       "auroc": float(roc_auc_score(y, pu)), **{f"tpr_{g}": v for g, v in gen_u.items()}}]
                     ).to_csv(out / "univfd_repro.csv", index=False)
        univfd_line = ["\nUNIVFD REPRODUCTION (released head, 224 center crop, threshold 0.5; Table I reports 67.8 on all 9 generators)",
                       f"  bAcc {(tpr_u + tnr_u) / 2:.4f}  TPR {tpr_u:.4f}  TNR {tnr_u:.4f}  AUROC {roc_auc_score(y, pu):.4f}",
                       "  TPR per generator: " + ", ".join(f"{g} {v:.3f}" for g, v in gen_u.items())]

    # ---------- per-run metrics, per-generator TPR, recompression shifts
    rows, gen_rows, rec_rows, preds = [], [], [], {}
    for f in sorted((out / "preds").glob(f"{args.backbone}_*.npz")):
        name = f.stem
        m = metrics_of(name)
        z = np.load(f)
        preds[name] = {k: z[k] for k in z.files}
        mm, pg = run_metrics(y, z["orig"], gen, m["val_best_threshold"])
        rows.append({"name": name, "strategy": m["strategy"], "version": m["version"], "lr": m["lr"], "seed": m["seed"], **mm})
        gen_rows += [{"name": name, "strategy": m["strategy"], "version": m["version"], "lr": m["lr"], "seed": m["seed"],
                      "generator": g, "tpr": v} for g, v in pg.items()]
        gen_rows.append({"name": name, "strategy": m["strategy"], "version": m["version"], "lr": m["lr"], "seed": m["seed"],
                         "generator": "authentic (TNR)", "tpr": mm["tnr"]})
        for q in QUALITIES:
            if f"q{q}" in z.files:
                for c, lab in ((0, "authentic"), (1, "synthetic")):
                    rec_rows.append({"name": name, "strategy": m["strategy"], "version": m["version"], "seed": m["seed"],
                                     "q": q, "class": lab, "s": float((z[f"q{q}"][y == c] - z["orig"][y == c]).mean())})
    R = pd.DataFrame(rows)
    R.to_csv(out / "metrics_runs.csv", index=False)
    pd.DataFrame(gen_rows).to_csv(out / "metrics_per_generator.csv", index=False)
    rec = pd.DataFrame(rec_rows)
    rec.to_csv(out / "recompression.csv", index=False)

    # ---------- contrasts at the selected LRs, per seed (III-A)
    def cell(strategy, version, metric="bacc"):
        lr = sel[strategy]
        d = R[(R.strategy == strategy) & (R.version == version) & np.isclose(R.lr, lr)].set_index("seed")[metric]
        return d.reindex(seeds)

    lines, crow = [], []
    for metric, label in (("bacc", "balanced accuracy @0.5 (primary)"), ("bacc_valthr", "balanced accuracy @ validation threshold"),
                          ("auroc", "AUROC"), ("tnr", "TNR (authentic)"), ("tpr", "TPR (synthetic)")):
        M = {(s, v): cell(s, v, metric) for s in ("lp", "lora") for v in versions}
        if any(c.isna().any() for c in M.values()):
            lines.append(f"[{label}] incomplete: some runs are missing")
            continue
        contr = {"Delta_LP": M["lp", "d_align"] - M["lp", "d_bias"],
                 "Delta_LoRA": M["lora", "d_align"] - M["lora", "d_bias"]}
        contr["I"] = contr["Delta_LoRA"] - contr["Delta_LP"]
        contr["R"] = M["lp", "d_align"] - M["lora", "d_align"]
        lines.append(f"\n[{label}]  cells (mean over {len(seeds)} seeds):  " +
                     ", ".join(f"{s.upper()}/{v}: {M[s, v].mean():.4f}" for s in ("lp", "lora") for v in versions))
        for k, v in contr.items():
            mean, sd, lo, hi, mde = t_interval(v.to_numpy())
            cat = category(mean, lo, hi)
            crow.append({"metric": metric, "contrast": k, "estimate": mean, "sd_over_seeds": sd, "ci_low": lo, "ci_high": hi,
                         "mde": mde, "category": cat, **{f"seed{s}": float(v[s]) for s in seeds}})
            lines.append(f"  {k:10s} {mean:+.4f}  95% t-CI [{lo:+.4f}, {hi:+.4f}]  sd {sd:.4f}  MDE {mde:.4f}  -> {cat}")
    pd.DataFrame(crow).to_csv(out / "contrasts.csv", index=False)

    # ---------- paired bootstrap over evaluation images (supporting analysis, III-D)
    strata = np.where(y == 0, "__authentic__", gen)
    boot_lines = []
    try:
        names = {(s, v): [R[(R.strategy == s) & (R.version == v) & np.isclose(R.lr, sel[s]) & (R.seed == sd)]["name"].iloc[0]
                          for sd in seeds] for s in ("lp", "lora") for v in versions}
        C = {k: np.stack([preds[n]["orig"] >= 0.5 for n in ns]) for k, ns in names.items()}   # [seeds, N] predicted fake
        correct = {k: np.where(y == 1, c, ~c).astype(np.float32) for k, c in C.items()}
        rng = np.random.default_rng(20261018)
        groups = [np.flatnonzero(strata == s) for s in np.unique(strata)]
        n_f, n_r = (y == 1).sum(), (y == 0).sum()
        res = {k: [] for k in ("Delta_LP", "Delta_LoRA", "I", "R")}
        for start in range(0, N_BOOT, 500):
            b = min(500, N_BOOT - start)
            W = np.zeros((b, len(y)), np.float32)
            for g in groups:
                idx = rng.integers(0, len(g), size=(b, len(g)))
                np.add.at(W, (np.repeat(np.arange(b), len(g)), g[idx].ravel()), 1)
            wf, wr = W * (y == 1), W * (y == 0)

            def mb(k):  # seed-averaged balanced accuracy for each resample
                cr = correct[k]
                return (0.5 * (wf @ cr.T / n_f + wr @ cr.T / n_r)).mean(1)
            M = {k: mb(k) for k in correct}
            dl, dr = M["lp", "d_align"] - M["lp", "d_bias"], M["lora", "d_align"] - M["lora", "d_bias"]
            res["Delta_LP"].append(dl); res["Delta_LoRA"].append(dr); res["I"].append(dr - dl)
            res["R"].append(M["lp", "d_align"] - M["lora", "d_align"])
        brow = []
        for k, v in res.items():
            v = np.concatenate(v)
            lo, hi = np.percentile(v, [2.5, 97.5])
            brow.append({"contrast": k, "mean": float(v.mean()), "pct_low": float(lo), "pct_high": float(hi)})
            boot_lines.append(f"  {k:10s} {v.mean():+.4f}  95% percentile [{lo:+.4f}, {hi:+.4f}]")
        pd.DataFrame(brow).to_csv(out / "bootstrap.csv", index=False)
    except (IndexError, KeyError) as e:
        boot_lines.append(f"  skipped: runs missing ({e})")

    # ---------- sensitivity: I for every LP lr x LoRA lr, first seed only (exploratory)
    srow = []
    f0 = R[R.seed == s0]
    for lr_lp in sorted(f0[f0.strategy == "lp"].lr.unique()):
        for lr_lo in sorted(f0[f0.strategy == "lora"].lr.unique()):
            def g(s, v, lr):
                r = f0[(f0.strategy == s) & (f0.version == v) & np.isclose(f0.lr, lr)]["bacc"]
                return float(r.iloc[0]) if len(r) else float("nan")
            dlp = g("lp", "d_align", lr_lp) - g("lp", "d_bias", lr_lp)
            dlo = g("lora", "d_align", lr_lo) - g("lora", "d_bias", lr_lo)
            srow.append({"lp_lr": lr_lp, "lora_lr": lr_lo, "Delta_LP": dlp, "Delta_LoRA": dlo, "I": dlo - dlp})
    S = pd.DataFrame(srow)
    S.to_csv(out / "sensitivity_I.csv", index=False)

    # ---------- recompression summary (mean over seeds, selected LRs)
    if len(rec):
        rs = rec.merge(R[["name", "lr"]], on="name")
        rs = rs[[np.isclose(r.lr, sel[r.strategy]) for r in rs.itertuples()]]
        rec_tab = rs.groupby(["strategy", "version", "class", "q"])["s"].mean().unstack("q").round(4)
    else:
        rec_tab = pd.DataFrame({"note": ["not computed (--no-recompression)"]})

    summary = [f"Protocol {frozen['fingerprint']} frozen {frozen['frozen_utc']}; LP lr {sel['lp']:g}, LoRA lr {sel['lora']:g}",
               f"Evaluation: {int((y == 1).sum())} synthetic ({', '.join(np.unique(gen[y == 1]))}), {int((y == 0).sum())} authentic",
               "\nCONTRASTS (seed-level 95% t-interval; categories per III-A, practical threshold 0.03)", *lines,
               "\nPAIRED BOOTSTRAP over evaluation images (seed-averaged, 10,000 resamples, stratified by class/generator)",
               *boot_lines,
               "\nSENSITIVITY: I on the first seed for every LP lr x LoRA lr (exploratory)",
               S.assign(lp_lr=S.lp_lr.map("{:g}".format), lora_lr=S.lora_lr.map("{:g}".format)).round(4).to_string(index=False),
               "\nRECOMPRESSION: mean posterior shift s(c,q), mean over seeds", rec_tab.to_string(), *univfd_line]
    (out / "summary.txt").write_text("\n".join(summary), encoding="utf-8")
    print("\n".join(summary))


if __name__ == "__main__":
    main()


In [ ]:
procs, t0 = [], time.time()
for g in range(max(1, N_GPUS)):
    cmd = [sys.executable, str(WORK / "eval_local.py"), "--runs", str(RUNS), "--data", str(DATA), "--out", str(OUT),
           "--batch", str(BATCH), "--workers", str(WORKERS), "--backbone", BACKBONE, *(["--no-recompression"] if not RECOMPRESSION else []),
           *(["--univfd-head", str(UNIVFD_HEAD)] if UNIVFD_HEAD else []), "--shard", str(g), "--nshards", str(max(1, N_GPUS)),
           "--skip-analysis"]
    logf = WORK / f"eval_gpu{g}.log"
    procs.append((g, logf, subprocess.Popen(cmd, env={**os.environ, "CUDA_VISIBLE_DEVICES": str(g), "PYTHONUNBUFFERED": "1"},
                                            stdout=open(logf, "w"), stderr=subprocess.STDOUT)))
last = 0
while any(p.poll() is None for *_, p in procs):
    time.sleep(30)
    if time.time() - last > 600:
        last = time.time()
        lines = [f"[gpu{g}] " + (logf.read_text().strip().splitlines() or [""])[-1] for g, logf, _ in procs]
        print("\n".join(lines), flush=True)
        notify(f"eval {(time.time() - t0) / 3600:.1f} h\n" + "\n".join(lines))
for g, logf, p in procs:
    print(f"--- gpu{g} exit {p.returncode}\n" + "\n".join(logf.read_text().splitlines()[-8:]))
assert all(p.returncode == 0 for *_, p in procs), "a worker failed: see the log above"

In [ ]:
# all predictions are cached now: this only computes the analysis
r = subprocess.run([sys.executable, str(WORK / "eval_local.py"), "--runs", str(RUNS), "--data", str(DATA), "--out", str(OUT),
                    "--batch", str(BATCH), "--workers", str(WORKERS), "--backbone", BACKBONE, *(["--no-recompression"] if not RECOMPRESSION else []),
                    *(["--univfd-head", str(UNIVFD_HEAD)] if UNIVFD_HEAD else [])], capture_output=True, text=True,
                   env={**os.environ, "CUDA_VISIBLE_DEVICES": "0"})
print(r.stdout[-20000:]); print(r.stderr[-3000:])
summary = (OUT / "summary.txt").read_text() if (OUT / "summary.txt").exists() else "no summary"
notify("EVAL DONE\n" + summary[:3000])